In [1]:
# Classificação Automática de Atividades Culturais e Educacionais
# 10 - Quando a API pede revisão humana
#
# Integrantes: Beatriz Aparecida de Mello Barbosa (10354067), Bruna Gonçalves Corte David (10425696),
#              Henrique Brainer Costa (10420717), João Pedro Queiroz de Andrade (10425822),
#              Júlia Andrade (10428513)
#
# Conteúdo: define, só com o treino, os dois sinais que fazem a API pedir revisão humana (a atividade
#           não se parece com nenhum grupo; o classificador está em dúvida), confere no gabarito se o
#           classificador da API erra mais nas atividades sinalizadas e mede o acerto dele em cada
#           categoria, com atenção às pequenas.
#
# Alterações:
#   27/10/2026 - Bruna - notebook novo, com os avisos de revisão e o acerto por categoria
#   16/11/2026 - Bruna - versão final

A API vai responder sozinha para atividades que ninguém do grupo viu, então ela precisa saber quando não
está segura e pedir que uma pessoa confira, do mesmo jeito que uma funcionária nova chama a chefe quando
fica em dúvida. As categorias vêm do classificador escolhido no notebook 09 (embeddings com regressão
logística), e encontramos dois motivos diferentes para desconfiar da resposta dele, cada um com o seu
sinal:

- **A atividade não se parece com nenhum grupo:** ela fica longe de tudo o que o classificador viu no
  treino. Medimos isso pela distância até os 16 grupos do notebook 07, porque o classificador sozinho
  não percebe esse caso: ele aprendeu fronteiras entre as categorias, e uma atividade muito diferente de
  tudo pode cair longe de qualquer fronteira e receber uma resposta confiante.
- **O classificador está em dúvida:** nenhuma categoria recebe uma probabilidade alta, ou seja, ele
  mesmo não sabe direito o que a atividade é.

Os dois limites saem do treino, com a mesma regra (fica sinalizada a atividade que estaria entre as 10%
menos seguras do treino naquele sinal), e o gabarito humano só aparece depois, para ver se o aviso pega
os erros do classificador.

In [2]:
# ====================================================================================================
#               CARREGAMENTO DAS ATIVIDADES, DO CLASSIFICADOR, DOS GRUPOS E DO GABARITO
# ====================================================================================================
import pandas as pd   # leitura e manipulação das tabelas
import numpy as np    # contas com vetores e matrizes
import json           # grava os limites num arquivo de texto
import joblib         # lê o classificador salvo no notebook 09

activities = pd.read_csv("../data/03_processed/07_activities_multilabel.csv", sep=";", encoding="utf-8-sig", keep_default_na=False)     # grupo de cada atividade e listas da régua, do notebook 07
activity_embeddings = np.load("../data/03_processed/03_embeddings.npy")                                                                 # vetores do notebook 03, na mesma ordem das linhas
gold_labels_table = pd.read_csv("../data/04_review/03_results/gold_labels.csv", sep=";", encoding="utf-8-sig", keep_default_na=False)   # o gabarito humano do notebook 08
api_classifier = joblib.load("../models/classifier.joblib")                                                                             # o classificador da API, salvo no notebook 09
cluster_centers = np.load("../models/cluster_centers.npy")                                                                              # os 16 centros, calculados só com o treino no notebook 07
cluster_names = pd.read_csv("../models/cluster_names.csv", sep=";", encoding="utf-8-sig").sort_values("cluster")                        # a categoria de cada grupo, na ordem dos centros

category_names = list(api_classifier["category_names"])   # as 13 categorias, na ordem das colunas do classificador
is_train = (activities["split"] == "train").values        # a atividade é do treino?

print("Atividades:", len(activities), "| Treino:", is_train.sum(), "| Gabarito:", len(gold_labels_table), "| Categorias:", len(category_names))

Atividades: 693 | Treino: 554 | Gabarito: 100 | Categorias: 13


## Sinal 1: a atividade não se parece com nenhum grupo

Para cada atividade, medimos a semelhança com o grupo mais parecido, como na régua do notebook 07, e
comparamos com a beirada da categoria desse grupo, que é o valor que 90% das atividades de treino
daquela categoria ultrapassam (experimento 6 do notebook 06). Abaixo da beirada, a atividade está mais
longe do grupo do que quase todos os membros dele.

In [3]:
# ====================================================================================================
#               SEMELHANÇA COM O GRUPO MAIS PARECIDO E A BEIRADA DE CADA CATEGORIA
# ====================================================================================================
normalized_embeddings = activity_embeddings / np.linalg.norm(activity_embeddings, axis=1, keepdims=True)                                                                  # vetores com comprimento 1
normalized_centers = cluster_centers / np.linalg.norm(cluster_centers, axis=1, keepdims=True)                                                                             # centros com comprimento 1
similarity_to_clusters = normalized_embeddings @ normalized_centers.T                                                                                                     # cosseno de cada atividade com cada um dos 16 centros
similarity_to_categories = np.array([similarity_to_clusters[:, (cluster_names["category"] == category_name).values].max(axis=1) for category_name in category_names]).T   # vale o grupo mais parecido de cada categoria

borders = {}                                                                                                                     # categoria -> beirada
for category_position, category_name in enumerate(category_names):
    members_in_train = is_train & (activities["category"] == category_name).values                                               # atividades de treino do grupo dessa categoria
    borders[category_name] = round(float(np.percentile(similarity_to_categories[members_in_train, category_position], 10)), 4)   # 90% delas ficam acima desse valor

closest_category = similarity_to_categories.argmax(axis=1)                                                           # a categoria do grupo mais parecido com cada atividade
closest_similarity = similarity_to_categories.max(axis=1)                                                            # a semelhança com ele
does_not_fit = closest_similarity < np.array([borders[category_names[position]] for position in closest_category])   # sinal 1: abaixo da beirada

print("Treino sinalizado por não se parecer com nenhum grupo:", does_not_fit[is_train].sum(), "de", is_train.sum())
pd.Series(borders)

Treino sinalizado por não se parecer com nenhum grupo: 60 de 554


Artes Visuais                        0.6571
Artesanato                           0.6187
Atividade Física e Bem-Estar         0.6547
Cinema                               0.5352
Cultura e Identidade                 0.6757
Esporte                              0.6382
Infância e Brincadeiras              0.6862
Literatura e Leitura                 0.6635
Música                               0.7080
Plantas e Alimentação                0.6163
Reciclagem e Sustentabilidade        0.6848
Shows, Espetáculos e Performances    0.5781
Sociedade e Debates                  0.5852
dtype: float64

## Sinal 2: o classificador está em dúvida

O sinal olha a maior probabilidade que o classificador dá entre as 13 categorias. Para tirar o limite do
treino há um cuidado: o classificador da API aprendeu com as próprias atividades de treino, então nelas
ele responde com mais certeza do que responderia a uma atividade nova. Por isso o limite sai das
respostas da validação cruzada, com as mesmas 5 partes do notebook 09, em que cada atividade de treino é
respondida por um modelo que não a viu.

In [4]:
# ====================================================================================================
#           A MAIOR PROBABILIDADE DO CLASSIFICADOR E O LIMITE, TIRADO DA VALIDAÇÃO CRUZADA
# ====================================================================================================
from sklearn.model_selection import KFold   # as mesmas 5 partes do notebook 09
from sklearn.base import clone              # uma cópia do classificador, com o mesmo botão, ainda sem treinar

train_activities = activities[is_train].reset_index(drop=True)                                                                                         # as 554 de treino, na mesma ordem do notebook 09
train_embeddings = activity_embeddings[train_activities["activity_id"].values]                                                                         # os vetores delas
train_targets = np.array([[int(category_name in labels.split(" | ")) for category_name in category_names] for labels in train_activities["labels"]])   # o que o classificador aprendeu: as listas da régua

out_of_fold_probabilities = np.zeros(train_targets.shape)                                                                      # probabilidades dadas a atividades de treino que o modelo não viu
for rows_to_learn, rows_to_answer in KFold(n_splits=5, shuffle=True, random_state=42).split(train_activities["text"].values):
    fold_classifier = clone(api_classifier["classifier"]).fit(train_embeddings[rows_to_learn], train_targets[rows_to_learn])   # aprende com 4 partes
    out_of_fold_probabilities[rows_to_answer] = fold_classifier.predict_proba(train_embeddings[rows_to_answer])                # responde a parte de fora

minimum_top_probability = round(float(np.percentile(out_of_fold_probabilities.max(axis=1), 10)), 4)   # 90% do treino tem a maior probabilidade acima disso
probabilities = api_classifier["classifier"].predict_proba(activity_embeddings)                       # o classificador da API em todas as atividades
in_doubt = probabilities.max(axis=1) < minimum_top_probability                                        # sinal 2: nenhuma categoria com probabilidade alta
train_in_doubt = out_of_fold_probabilities.max(axis=1) < minimum_top_probability                      # no treino, vale a resposta da validação cruzada

print("Limite da maior probabilidade:", minimum_top_probability)
print("Treino sinalizado por dúvida do classificador:", train_in_doubt.sum(), "de", len(train_activities))
print("O mesmo limite com o modelo que aprendeu com elas mesmas sinalizaria:", in_doubt[is_train].sum(), "de", is_train.sum())
print("Treino sinalizado por pelo menos um dos dois:", (does_not_fit[is_train] | train_in_doubt).sum(), "de", is_train.sum())

Limite da maior probabilidade: 0.443
Treino sinalizado por dúvida do classificador: 56 de 554
O mesmo limite com o modelo que aprendeu com elas mesmas sinalizaria: 0 de 554
Treino sinalizado por pelo menos um dos dois: 95 de 554


In [5]:
# ----------------------------------------------------------------------------------------------------
#                                      OS LIMITES VÃO PARA A API
# ----------------------------------------------------------------------------------------------------
review_limits = {"borders": borders, "minimum_top_probability": minimum_top_probability}   # os dois limites, escolhidos só no treino
limits_file = open("../models/review_limits.json", "w", encoding="utf-8")                  # junto do classificador, em models/
json.dump(review_limits, limits_file, ensure_ascii=False, indent=1)                        # grava os limites
limits_file.close()                                                                        # fecha o arquivo

print("Limites salvos em models/review_limits.json")

Limites salvos em models/review_limits.json


## O aviso pega os erros do classificador?

Se o aviso funciona, o classificador deve errar mais nas atividades sinalizadas que nas outras, e
medimos isso no gabarito de dois jeitos: se a 1ª categoria do classificador está entre as 1ªs válidas do
gabarito, e o F1 de cada atividade (1 quando a lista do classificador é igual à do gabarito, 0 quando
não têm nada em comum). A lista do classificador é montada como na API: entra toda categoria acima do
próprio corte, e, se nenhuma passar, entra a mais provável.

In [6]:
# ====================================================================================================
#                ACERTO DO CLASSIFICADOR NAS SINALIZADAS E NAS NÃO SINALIZADAS, NO GABARITO
# ====================================================================================================
def split_labels(labels):
    return [label for label in labels.split(" | ") if label != ""]   # texto "A | B" vira a lista [A, B]

def activity_f1(predicted_list, gold_list):
    if not predicted_list and not gold_list:
        return 1.0                                                                                  # as duas listas vazias: acertou
    return 2 * len(set(predicted_list) & set(gold_list)) / (len(predicted_list) + len(gold_list))   # 1 se iguais, 0 se nada em comum

predicted = (probabilities >= api_classifier["thresholds"]).astype(int)                                                                                          # entra toda categoria acima do próprio corte
without_any = predicted.sum(axis=1) == 0                                                                                                                         # atividades que ficaram sem nenhuma
predicted[np.where(without_any)[0], probabilities[without_any].argmax(axis=1)] = 1                                                                               # essas recebem a mais provável, como na API
classifier_labels = [[category_names[position] for position in np.argsort(-probabilities[row]) if predicted[row, position]] for row in range(len(activities))]   # a lista de cada atividade, da mais provável para a menos
position_by_id = {activity_id: position for position, activity_id in enumerate(activities["activity_id"])}                                                       # identificador -> linha nas matrizes e nas listas

gold_rows = []                                                                                                                                                                                                                                                                                                                                                                                                                                                    # uma linha por atividade do gabarito
for _, gold in gold_labels_table.iterrows():
    position = position_by_id[gold["activity_id"]]                                                                                                                                                                                                                                                                                                                                                                                                                # a linha da atividade
    classifier_list = classifier_labels[position]                                                                                                                                                                                                                                                                                                                                                                                                                 # a lista do classificador
    first_is_valid = bool(classifier_list) and classifier_list[0] in split_labels(gold["gold_first_categories"])                                                                                                                                                                                                                                                                                                                                                  # acertou o que a atividade é?
    gold_rows.append({"activity_id": gold["activity_id"], "title": gold["title"], "does_not_fit": does_not_fit[position], "in_doubt": in_doubt[position], "flagged": does_not_fit[position] or in_doubt[position], "first_category_hit": first_is_valid, "activity_f1": activity_f1(classifier_list, split_labels(gold["gold_labels"])), "gold_has_no_category": gold["gold_labels"] == "", "top_probability": round(float(probabilities[position].max()), 3)})   # sinais e acerto
gold_check = pd.DataFrame(gold_rows)                                                                                                                                                                                                                                                                                                                                                                                                                              # lista vira tabela

flag_summary = gold_check.groupby("flagged").agg(activities=("activity_id", "count"), first_category_hit_rate=("first_category_hit", "mean"), mean_activity_f1=("activity_f1", "mean")).round(3)   # acerto de cada grupo
print("1ª categoria do classificador entre as 1ªs válidas do gabarito:", gold_check["first_category_hit"].sum(), "de", len(gold_check))
first_category_errors = ~gold_check["first_category_hit"]                                                                                                                                          # o classificador errou a 1ª categoria
print("Erros de 1ª categoria:", first_category_errors.sum(), "| sinalizados pelo aviso:", (first_category_errors & gold_check["flagged"]).sum())
print("Sinalizadas com a lista igual à do gabarito:", (gold_check["flagged"] & (gold_check["activity_f1"] == 1)).sum())
print("Sinalizadas por não se parecer com nenhum grupo:", gold_check["does_not_fit"].sum(), "| por dúvida do classificador:", gold_check["in_doubt"].sum(), "| pelos dois:", (gold_check["does_not_fit"] & gold_check["in_doubt"]).sum())
flag_summary

1ª categoria do classificador entre as 1ªs válidas do gabarito: 63 de 100
Erros de 1ª categoria: 37 | sinalizados pelo aviso: 10
Sinalizadas com a lista igual à do gabarito: 4
Sinalizadas por não se parecer com nenhum grupo: 19 | por dúvida do classificador: 12 | pelos dois: 8


,activities,first_category_hit_rate,mean_activity_f1
flagged,,,
False,77,0.649,0.683
True,23,0.565,0.519


In [7]:
# ----------------------------------------------------------------------------------------------------
#                                     AS SINALIZADAS, UMA POR UMA
# ----------------------------------------------------------------------------------------------------
gold_check[gold_check["flagged"] | gold_check["gold_has_no_category"]][["activity_id", "title", "does_not_fit", "in_doubt", "first_category_hit", "activity_f1", "top_probability", "gold_has_no_category"]]

,activity_id,title,does_not_fit,in_doubt,first_category_hit,activity_f1,top_probability,gold_has_no_category
4,41,FESTIVAL DE ARTES DA FÁBRICA DE CULTURA EM TAI...,True,False,False,0.000000,0.653,False
10,63,OUTUBRO ROSA: SAÚDE DA MULHER EM PAUTA,True,True,False,0.000000,0.087,False
11,69,MANTO DO INVISÍVEL: MEMÓRIAS EM PRODUÇÕES TÊXTEIS,False,True,False,0.000000,0.169,False
12,70,ANTES DA ANIMAÇÃO: BRINQUEDOS ÓPTICOS – COM EQ...,False,True,True,0.666667,0.339,False
13,72,DIA DO SACI | JAXY JATERÊ – CONFECÇÃO DE MARCA...,True,False,False,0.400000,0.741,False
22,132,Mostra Tecnologias de Todos os Tempos - 80 anos,False,True,True,1.000000,0.076,False
26,155,"Mural ""Memórias Uchinanchu""",True,True,False,0.666667,0.155,False
37,223,Luta olímpica com Aline Silva,True,False,True,0.500000,0.444,False
42,247,Festival de futsal para mulheres,True,True,True,1.000000,0.416,False
44,249,Festival de Futebol Feminino,True,False,True,0.500000,0.992,False


**O que deu.** O classificador acerta a 1ª categoria em 63 das 100 atividades do gabarito, e o aviso
sinalizou 23 delas, 19 por não se parecerem com nenhum grupo, 12 por dúvida do classificador e 8 pelos
dois. Para ver se o aviso funciona, comparamos as 23 sinalizadas com as 77 outras de dois jeitos: na
lista inteira, as sinalizadas ficam mais longe do gabarito (F1 médio por atividade de 0,52, contra 0,68
nas outras), e na 1ª categoria a diferença é pequena, com o classificador acertando em 13 das 23
sinalizadas (57%), contra 50 das 77 outras (65%).

Das três atividades que ficaram sem categoria no gabarito, o aviso pegou duas pelos dois sinais, o Curso
de Atuações (367) e O deserto fértil da escrita (549). A terceira, Conexão Jovem (456), passou sem
aviso, porque se parece com um grupo e o classificador dá a ela uma categoria com 0,64 de probabilidade,
acima do limite de dúvida.

O aviso indica onde a resposta tende a ser pior, mas está longe de pegar todos os erros: das 37
atividades em que o classificador erra a 1ª categoria, ele sinaliza 10, e entre as 23 sinalizadas há 4
em que a lista do classificador é igual à do gabarito. Com 100 atividades, a diferença na 1ª categoria é
pequena demais para ser confirmada, e o que dá para dizer é que o aviso ajuda a escolher o que conferir
primeiro, sem dispensar a conferência do resto.

## O acerto de cada categoria

O parecer da Parte 2 pediu atenção às categorias menos representadas, então medimos o acerto do
classificador em cada categoria no gabarito: a precisão (das vezes em que o classificador deu a
categoria, quantas o gabarito confirma) e a cobertura (das atividades que têm a categoria no gabarito,
quantas o classificador achou).

In [8]:
# ====================================================================================================
#                        PRECISÃO, COBERTURA E F1 DE CADA CATEGORIA NO GABARITO
# ====================================================================================================
category_rows = []                                                                                                                                                                                            # uma linha por categoria
for category_name in category_names:
    given_by_classifier = 0                                                                                                                                                                                   # quantas vezes o classificador deu a categoria
    in_gold = 0                                                                                                                                                                                               # quantas atividades têm a categoria no gabarito
    both = 0                                                                                                                                                                                                  # quantas vezes as duas coisas acontecem juntas
    for _, gold in gold_labels_table.iterrows():
        classifier_has = category_name in classifier_labels[position_by_id[gold["activity_id"]]]                                                                                                              # o classificador deu?
        gold_has = category_name in split_labels(gold["gold_labels"])                                                                                                                                         # o gabarito tem?
        given_by_classifier += classifier_has                                                                                                                                                                 # soma
        in_gold += gold_has                                                                                                                                                                                   # soma
        both += classifier_has and gold_has                                                                                                                                                                   # soma
    precision = both / given_by_classifier if given_by_classifier else 0                                                                                                                                      # das que o classificador deu, quantas certas
    coverage = both / in_gold if in_gold else 0                                                                                                                                                               # das que existem, quantas achadas
    f1 = 2 * precision * coverage / (precision + coverage) if precision + coverage else 0                                                                                                                     # equilíbrio das duas
    category_rows.append({"category": category_name, "in_gold": in_gold, "given_by_classifier": given_by_classifier, "precision": round(precision, 2), "coverage": round(coverage, 2), "f1": round(f1, 2)})   # guarda a linha
category_table = pd.DataFrame(category_rows).sort_values("in_gold")                                                                                                                                           # as menores primeiro
category_table

,category,in_gold,given_by_classifier,precision,coverage,f1
9,Plantas e Alimentação,3,3,0.67,0.67,0.67
10,Reciclagem e Sustentabilidade,5,4,0.75,0.60,0.67
3,Cinema,7,9,0.78,1.00,0.88
7,Literatura e Leitura,9,12,0.58,0.78,0.67
0,Artes Visuais,11,14,0.50,0.64,0.56
1,Artesanato,14,9,0.78,0.50,0.61
6,Infância e Brincadeiras,15,15,0.60,0.60,0.60
8,Música,18,8,1.00,0.44,0.62
11,"Shows, Espetáculos e Performances",18,20,0.80,0.89,0.84
4,Cultura e Identidade,19,7,0.71,0.26,0.38


**O que deu.** As duas categorias menores foram razoavelmente bem: Plantas e Alimentação achou 2 das 3
atividades do gabarito, com um alarme falso, e Reciclagem e Sustentabilidade achou 3 das 5, também com
um alarme falso. Com tão poucas atividades, esses números servem de sinal, não de medida.

As categorias que preocupam são as que nós usamos muito como categoria seguinte e o classificador quase
não dá. Cultura e Identidade (19 atividades no gabarito) foi dada só 7 vezes, com 5 acertos, e Atividade
Física e Bem-Estar (23 no gabarito) só 6 vezes, com 4 acertos, ficando com F1 de 0,38 e 0,28. Sociedade
e Debates vai pelo mesmo caminho em menor grau: o classificador acerta todas as 8 vezes em que a dá, mas
acha só 8 das 21.

Não criamos regra extra para as categorias pequenas, porque 3 e 5 atividades são poucas para decidir um
corte. Nessas três categorias o problema não é a API dar a categoria errada, já que, quando dá, acerta
na maior parte das vezes, e sim deixar de dar. Como o classificador aprendeu com as listas da régua, que
também vai mal nessas categorias (notebook 09), o caminho mais direto para melhorar é ele aprender com
as correções que fazemos quando a API pede revisão.